# Lab 14: How to search your documents with LlamaIndex

**Notebook 2 of 2: LlamaIndex at the Chinook support desk (challenge solved)**

## Goal

Put LlamaIndex to work on the files of the Chinook music shop's support desk: 20 policies (the shop's rules) and 16
support tickets (what happened to one customer). A rep asks:

> *Ticket 06's replacement arrived warped too. What do we offer now, and how long does the refund take?*

The answer is in two policy files, and the question names neither of them. Ticket 06 itself points to one of them: one
line names the policy it was handled under. We hand the search to an agent in two ways, one run each, and read what
each tool hands back: the query-engine tool a sentence that another model call wrote about the files, the retriever
tool the files' text. What we check is whether the ticket's line that names the policy reaches the agent. The result
that held on every call is in Part C, where each tool is called once by hand on ticket 06; the agents in Parts D and E
write their own searches, so their runs vary.

## What you will learn

- How to index the shop's policy and ticket files with `SimpleDirectoryReader` and `VectorStoreIndex`.
- How to check a query engine's answer against its `source_nodes` and the prompt it filled in, and find which file was
  missing.
- How to tell from one call of each tool whether a **pointer**, a line in one file that names another file, reaches
  the agent.
- How to run a `FunctionAgent` with a query-engine tool and with a retriever tool, and read every search it made and
  every file each search found.
- How to count every model call of a run, split a long document into nodes, and watch an agent's events (Going
  further).

## From notebook 1 to the shop's files

Notebook 1 met LlamaIndex's objects on a folder of notes, in this order:

1. `Settings` holds the LLM and the embedding model; we set both from `models.env`.
2. `SimpleDirectoryReader` reads a folder into `Document`s; `keep_name` keeps only each file's name as metadata.
3. `VectorStoreIndex.from_documents` embeds them; `index.as_retriever()` returns the 2 nearest nodes with scores.
4. `index.as_query_engine()` retrieves, makes one model call, and returns an answer with its `source_nodes`.
5. `QueryEngineTool` and `RetrieverTool` wrap the query engine or the retriever as a tool for a `FunctionAgent`.

Now we take it seriously: a rep will repeat the answer to a customer, so a missing rule matters. This notebook
restates everything it uses, so it runs on its own.

## Setup

The first cell is the same in every lab: it finds the repository folder, reads the model names from `models.env`, and
takes the key from `.env` without printing it.

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


Then the two models LlamaIndex will use, from `models.env`. Without this cell, `Settings` would fall back to
LlamaIndex's own default models (notebook 1, Part A).

In [2]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = OpenAI(model=MODEL, reasoning_effort=EFFORT)
Settings.embed_model = OpenAIEmbedding(model=settings["EMBEDDING_MODEL"])
print("LLM:", Settings.llm.model, "| embedding:", Settings.embed_model.model_name)

LLM: gpt-6-luna | embedding: text-embedding-3-small


---

## Part A: Index the shop's policy and ticket files

The shop's files are short markdown files in `data/corpus/`, read in place and never changed. We read them with
`SimpleDirectoryReader` and the same `keep_name` function as in notebook 1, so that no model reads the full path of a
file on your computer.

In [3]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

CORPUS = ROOT / "data" / "corpus"


def keep_name(path):
    return {"file_name": Path(path).name}


documents = SimpleDirectoryReader(CORPUS, file_metadata=keep_name).load_data()
names = [document.metadata["file_name"] for document in documents]
print(len(documents), "documents:", sum(n.startswith("policy") for n in names), "policies and",
      sum(n.startswith("ticket") for n in names), "tickets")

index = VectorStoreIndex.from_documents(documents)
print(type(index).__name__, "with", len(index.docstore.docs), "nodes")

36 documents: 20 policies and 16 tickets


VectorStoreIndex with 36 nodes


36 documents, one per file, and 36 nodes: every file is a few short paragraphs, so none is split.

Here is the ticket the rep's question starts from, as it is on disk:

In [4]:
print((CORPUS / "ticket_06_germany_warped_vinyl.md").read_text())

# Ticket 06 — Germany, warped vinyl

Date: 2012-07-18
Customer: billing country Germany

Vinyl arrived warped. Customer sent a photo. Damaged-media policy: replace once at no charge.

Status: resolved. Replacement shipped 2012-07-20.



Ticket 06 is a record: a vinyl arrived warped in 2012, and the shop sent one free replacement. Now the replacement has
arrived warped too. A right answer needs two facts, and neither is in the ticket:

1. **What to offer now:** a refund instead of a third copy. It is in `policy_damaged_media.md`: *"If the same title
   arrives damaged twice, offer a refund instead of a third copy."*
2. **How long the refund takes:** 5 to 10 business days after support approves it. It is in `policy_refunds.md`.

Notice the line *"Damaged-media policy: replace once at no charge"* in the ticket. It is a **pointer**: it names the policy
that holds fact 1, without quoting its rule for a second damaged copy. A reader who follows that name finds the rule. Keep this line
in mind for the rest of the notebook.

**Checkpoint A.** 36 documents (20 policies, 16 tickets) and 36 nodes; ticket 06 holds the line *Damaged-media
policy: replace once at no charge*.

---

## Part B: Ask a query engine and check its answer against its sources

First the shortest way: a query engine with the defaults, which retrieves the 2 nearest files and makes one model call.

In [5]:
QUESTION = "Ticket 06's replacement arrived warped too. What do we offer now, and how long does the refund take?"

query_engine = index.as_query_engine()
response = query_engine.query(QUESTION)

print(response)

Since the vinyl was already replaced once at no charge, offer a refund to the original payment method. After approval, it takes 5 to 10 business days.


In our run: *Since the vinyl was already replaced once at no charge, offer a refund to the original payment method. After approval, it takes 5 to 10 business days.* It reads as right. Before we trust it, we check it.

To check answers the same way all through the notebook, we write down in code what a right answer contains. `check`
looks for fact 1 (the word *refund* together with *instead*, *rather than*, *third* or *twice*, which say that it
replaces a further copy) and fact 2 (*5 to 10 business days*, with *to* or a dash). `re.search` finds a pattern
anywhere in the text; `\s*` allows spaces. It is a keyword check, so we read every answer it marks as well.

In [6]:
import re


def check(answer):
    text = answer.lower()
    rule = "refund" in text and any(word in text for word in ["instead", "rather than", "third", "twice"])
    days = re.search(r"5\s*(to|-|–)\s*10 business days", text) is not None
    print("✅" if rule else "❌", "refund instead of a third copy")
    print("✅" if days else "❌", "5 to 10 business days")


check(str(response))

❌ refund instead of a third copy
✅ 5 to 10 business days


Fact 2 passes. Fact 1 fails: the answer offers a refund, but never says that the refund replaces a third copy, which
is what the rule says. Even a ✅ on fact 1 could not tell us whether the rule came from a file or from the model's own
inference. Here the offer is right, but the part only the rule gives, a refund in place of a third copy, is missing: a
sign that the rule may not have been in what the model read. The sources show whether the rule was there:

In [7]:
for source in response.source_nodes:
    print(f"{source.score:.4f}  {source.node.metadata['file_name']}")

0.6123  ticket_06_germany_warped_vinyl.md
0.4844  policy_refunds.md


Ticket 06 (0.6123), which the question names, and `policy_refunds.md` (0.4844), which holds fact 2. `policy_damaged_media.md`, the file with the rule for a second damaged copy, is not among them: the default top k is 2, and it ranks lower.

What did the model read? `query_engine.get_prompts()` returns the engine's prompt templates by name, and the one used
for a question is `response_synthesizer:text_qa_template`. The engine filled it in with these two files. We fill in the same template with the same two sources and the question, and print the result:
`get_content()` returns a node's text, and `format()` puts the values where `{context_str}` and `{query_str}` are.

In [8]:
template = query_engine.get_prompts()["response_synthesizer:text_qa_template"]
context = "\n\n".join(source.node.get_content() for source in response.source_nodes)

print(template.format(llm=Settings.llm, context_str=context, query_str=QUESTION))

system: You are an expert Q&A system that is trusted around the world.
Always answer the query using the provided context information, and not prior knowledge.
Some rules to follow:
1. Never directly reference the given context in your answer.
2. Avoid statements like 'Based on the context, ...' or 'The context information ...' or anything along those lines.
user: Context information is below.
---------------------
# Ticket 06 — Germany, warped vinyl

Date: 2012-07-18
Customer: billing country Germany

Vinyl arrived warped. Customer sent a photo. Damaged-media policy: replace once at no charge.

Status: resolved. Replacement shipped 2012-07-20.

# Refunds

Refunds go back to the original payment method. They take 5 to 10 business days after support approves the request.

Partial refunds are allowed when only some tracks on a multi-disc set are defective.

We do not refund because a customer changed their mind about a digital purchase.

Duplicate charges: if two invoices exist for the s

This is the prompt in front of the model: LlamaIndex's system message, then ticket 06 and the refunds policy, then the
question. The damaged-media policy is not in it. So the refund in the answer is the model's own inference, from the ticket's *replace once* and the refunds file. Here the inference happens to match the rule. Had the rule said *store credit*, the answer would read just as well and be wrong. The sources are how we can tell.

**Checkpoint B.** One answer, written from ticket 06 and `policy_refunds.md`; `policy_damaged_media.md` is not among
the sources and not in the prompt. The check fails fact 1 and passes fact 2.

---

## Part C: Call both tools by hand on ticket 06

An agent would not stop at one query: it can search again for what is still open. So before any agent runs, we look
at what each kind of tool would hand it when it searches for ticket 06. Both tools get the same name and description,
so the only difference is what comes back.

Here no agent chooses the searches: each tool is called once by hand on the same input. `tool.call(input=...)` runs a
tool as an agent would and returns a `ToolOutput`; its `.content` is the text the agent reads.

In [9]:
from llama_index.core.tools import QueryEngineTool, RetrieverTool

DESCRIPTION = "Search the shop's policy and support ticket files."

qe_tool = QueryEngineTool.from_defaults(query_engine=query_engine, name="search", description=DESCRIPTION)
rt_tool = RetrieverTool.from_defaults(retriever=index.as_retriever(), name="search", description=DESCRIPTION)

print("QUERY-ENGINE TOOL")
print(qe_tool.call(input="ticket 06 warped vinyl").content)
print()
print("RETRIEVER TOOL")
print(rt_tool.call(input="ticket 06 warped vinyl").content)

QUERY-ENGINE TOOL


Ticket 06: The warped vinyl was replaced at no charge. The replacement shipped on 2012-07-20.

RETRIEVER TOOL


# Ticket 06 — Germany, warped vinyl

Date: 2012-07-18
Customer: billing country Germany

Vinyl arrived warped. Customer sent a photo. Damaged-media policy: replace once at no charge.

Status: resolved. Replacement shipped 2012-07-20.

# Ticket 11 — Ireland, exchange for a different pressing

Date: 2012-10-29
Customer: billing country Ireland

Customer wants to exchange an unopened CD for the vinyl of the same album. Vinyl is 8 dollars more.

Status: open. Waiting on the customer to accept the 8 dollar difference.




Read the two side by side:

- **The query-engine tool** returned one sentence: *Ticket 06: The warped vinyl was replaced at no charge. The replacement shipped on 2012-07-20.* Every word of it is true, and the words *Damaged-media policy* are gone.
- **The retriever tool** returned two tickets as they are on disk: ticket 06, with its line *Damaged-media policy:
  replace once at no charge*, and ticket 11, which is about something else.

Both tools search with the same top k of 2, so the engine retrieved the same two tickets the retriever tool printed,
and its model call was given ticket 06 with its line included. The sentence answers the query, *what happened in
ticket 06*. The policy's name is not part of that answer, so the model that wrote the sentence left it out. It did so
in all 10 calls we made.

**Checkpoint C.** The query-engine tool's sentence about ticket 06 does not name the damaged-media policy; the
retriever tool's text holds the ticket's *Damaged-media policy* line.

---

## Part D: Run the agent with the query-engine tool

Now the agent. Its prompt is this lab's own: answer from the files, search again while part of the question is open,
and say so when the files do not answer. We run it once and print every search it made. `qe_agent.run(user_msg=...)`
starts a run, and `await` waits for it to finish (a notebook lets us write `await` directly in a cell). The agent
writes its own searches, so one run can differ from the next; Part C's result held on every call.

`FunctionAgent` gives us `result.tool_calls`, with the query the model wrote (`tool_kwargs`) and the text the tool
handed back (`tool_output.content`). One more thing helps here: which files each search found, though the agent is
never given their names. `files_found` reads them from `tool_output.raw_output`, the tool's raw result. For a query-engine
tool that is a `Response`, whose `source_nodes` we read; for a retriever tool it is the list of nodes itself, which
`hasattr(raw, "source_nodes")` tells apart.

In [10]:
from llama_index.core.agent.workflow import FunctionAgent

PROMPT = (
    "You answer a support rep's questions from the shop's policy and ticket files. "
    "Use the search tool, and search again while part of the question is still open. "
    "If the files do not answer, say so."
)


def files_found(call):
    raw = call.tool_output.raw_output
    found = raw.source_nodes if hasattr(raw, "source_nodes") else raw
    return [item.node.metadata["file_name"] for item in found]


qe_agent = FunctionAgent(tools=[qe_tool], system_prompt=PROMPT)
qe_result = await qe_agent.run(user_msg=QUESTION)

for call in qe_result.tool_calls:
    print("search:", call.tool_kwargs["input"])
    print("found: ", files_found(call))
    print("read:  ", call.tool_output.content)
    print()
print("answer:", qe_result.response.content)
print()
check(qe_result.response.content)

search: Ticket 06 replacement arrived warped what offer now refund processing time policy
found:  ['ticket_06_germany_warped_vinyl.md', 'ticket_07_uk_preorder_delay.md']
read:   The damaged-media policy covers one no-charge replacement. Since that replacement also arrived warped, no further remedy or refund-processing time is specified.

search: Ticket 06 warped replacement customer support ticket resolution offer refund timeline
found:  ['ticket_06_germany_warped_vinyl.md', 'ticket_07_uk_preorder_delay.md']
read:   The warped vinyl was replaced once at no charge. The replacement shipped on July 20, 2012; no refund timeline was recorded.

answer: Ticket 06 already received the one no-charge replacement allowed by the damaged-media policy. The files don’t say what further remedy we can offer or how long a refund would take, so we’ll need to confirm those details before promising either.

❌ refund instead of a third copy
❌ 5 to 10 business days


In our run the agent searched twice, and both searches found ticket 06 with ticket 07, never the damaged-media policy. Read the two sentences it was given. The first kept the policy's name and then added a claim: *no further remedy or refund-processing time is specified*. The second dropped the name: *The warped vinyl was replaced once at no charge ... no refund timeline was recorded.* The engine's model call was given two tickets, not all 36 files, and the agent had no file to check those claims against. It answered that the files do not say what to offer or how long a refund takes, and both facts fail.

The agent read sentences, never files: the `found:` lines are for us, and the agent was given only the `read:` lines. The rule for a second damaged copy was one search away, and nothing the agent read pointed it there.

This run missed, which is the more common result: with this prompt, the query-engine-tool agent reached the rule in 8 of our 18 runs, and the retriever-tool agent (Part E) in 8 of 8. The question's own words (*replacement*, *warped*, *refund*) are close in meaning to the damaged-media policy, which is why a query built from them sometimes finds it without the pointer. The saved run of this cell in `14b_llamaindex_at_the_desk.ipynb` is one of the hits: its first, long search found the policy file itself.

**Checkpoint D.** In our run, two searches, each finding ticket 06 and ticket 07; sentences saying no further remedy or refund time is specified; both facts fail.

---

## Part E: Run the same agent with the retriever tool

The same model, prompt and question; only the tool changes. The retriever tool returns whole files, so we print the
files each search found and leave their text out.

In [11]:
rt_agent = FunctionAgent(tools=[rt_tool], system_prompt=PROMPT)
rt_result = await rt_agent.run(user_msg=QUESTION)

for call in rt_result.tool_calls:
    print("search:", call.tool_kwargs["input"])
    print("found: ", files_found(call))
    print()
print("answer:", rt_result.response.content)
print()
check(rt_result.response.content)

search: Ticket 06 replacement arrived warped too: what now offered, and how long refund takes; find ticket 06 and relevant refund policy.
found:  ['ticket_06_germany_warped_vinyl.md', 'policy_refunds.md']

search: Damaged media policy if a one-time replacement also arrives damaged/warped; remedies after replacement, refund eligibility/approval process. Germany vinyl damaged goods.
found:  ['ticket_06_germany_warped_vinyl.md', 'policy_damaged_media.md']

search: refund timeline 5 to 10 business days after approval original payment method; policy exactly
found:  ['policy_refunds.md', 'ticket_07_uk_preorder_delay.md']

answer: Because the same title arrived damaged twice, offer a **refund instead of sending a third copy**. Refunds go to the original payment method and take **5–10 business days after support approves the request**.

✅ refund instead of a third copy
✅ 5 to 10 business days


In our run the first search found ticket 06 and `policy_refunds.md`. The agent read the ticket's text, with its line *Damaged-media policy: replace once at no charge*, and its second query was about the *damaged media policy*, which found `policy_damaged_media.md`. A third search looked up the refund time. Both facts pass. This is the second search done the way a person would: a policy named in a ticket, then looked up.

We cannot see why the model chose its words, but *damaged media* and *policy* are the ticket's words, not the question's. What the retriever tool makes sure of is that the pointer, once its ticket is found, reaches the agent.

**Checkpoint E.** In our run, three searches; the second found `policy_damaged_media.md` after the ticket's pointer; both facts pass.

### What each agent read

| | Query-engine tool | Retriever tool |
|---|---|---|
| What a search hands back | a sentence another model call wrote | the files' text |
| Searches in this run | 2 | 3 |
| Model calls inside the tool | one per search | none |
| Did the damaged-media policy reach the agent in this run? | no: no search found it, and no sentence pointed to it | yes, as the file's text: the agent followed the ticket's pointer |
| Both facts in this run | no | yes |
| Runs that reached the policy, across our runs | 8 of 18 | 8 of 8 |

One run is not enough to compare the two tools: across runs the query-engine-tool agent reached the policy in fewer
than half, and in this run it missed. Part C shows why: on every call we made, the sentence about ticket 06 left out
the pointer.

**Key lesson:** a query-engine tool puts a second model call between the files and the agent. That call writes its
own answer to the agent's query, and whatever it leaves out, the agent never sees: in Part C, the ticket's pointer to
the policy. A retriever tool hands over the evidence itself and lets the agent follow it. A query-engine tool is worth its
extra calls when the agent should not read the raw text (very long documents, or many of them); then test it on
questions that need a second search.

---

## What you can do now

- Index a folder of business files with `SimpleDirectoryReader` and `VectorStoreIndex`, keeping only the file name as
  metadata (Part A).
- Check a query engine's answer in code and against its `source_nodes`, and print the prompt it filled in (Part B).
- Call a query-engine tool and a retriever tool by hand and see whether a pointer in a file reaches the agent (Part C).
- Run a `FunctionAgent` with either tool, and read every search, the files it found and what the agent read (Parts D
  and E).
- Say which tool to give an agent, and when the other one is worth its extra model calls (Part E).

## Challenge

1. **A gift-card question with no agent.** A rep asks *Do gift cards expire, and can a customer exchange one for
   cash?* We know the topic in advance, so build a query engine limited to the one file: `index.as_query_engine(filters=...)`
   with `MetadataFilters(filters=[MetadataFilter(key="file_name", value="policy_gift_cards.md")])` (both from
   `llama_index.core.vector_stores`). Print the answer and its sources. Expected: one model call and one source,
   `policy_gift_cards.md`; the answer says the cards do not expire and that they are exchanged for cash only where the
   law requires it.
2. **A second search on another ticket.** Ask both agents, once each: *Ticket 10's customer is asking again. Can we
   add the student discount to her invoice now, and would it cover a digital album?* Ticket 10 says the discount is
   applied at checkout only; only `policy_student_discount.md` says digital albums are not discounted. Print each run's
   searches with `files_found`. Expected: both agents answer both parts: no, the discount is applied at checkout only and is not added once the invoice exists; and no, digital albums are not discounted. In our run each agent searched once, and that one search found `policy_student_discount.md` beside ticket 10, because the question already names the discount. Then say why the ticket's pointer mattered for ticket 06 and not here.

The worked solution, run with its outputs, is in `solved/14b_llamaindex_at_the_desk_solution.ipynb`.

### Solution 1: a gift-card question with no agent

In [12]:
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters

gift_cards_only = MetadataFilters(filters=[MetadataFilter(key="file_name", value="policy_gift_cards.md")])
gift_engine = index.as_query_engine(filters=gift_cards_only)

gift = gift_engine.query("Do gift cards expire, and can a customer exchange one for cash?")
print(gift)
print([source.node.metadata["file_name"] for source in gift.source_nodes])

Gift cards do not expire. They can’t be exchanged for cash unless the law requires it.
['policy_gift_cards.md']


One model call, one source: `policy_gift_cards.md`. The answer has both points: gift cards do not expire, and they are exchanged for cash only where the law requires it. The filter is our code's decision, made before any ranking: a search limited to one file cannot return another.

### Solution 2: a second search on another ticket

In [13]:
TICKET_10 = (
    "Ticket 10's customer is asking again. Can we add the student discount to her invoice now, "
    "and would it cover a digital album?"
)

for label, agent in [("query-engine tool", qe_agent), ("retriever tool", rt_agent)]:
    result = await agent.run(user_msg=TICKET_10)
    print("---", label)
    for call in result.tool_calls:
        print("search:", call.tool_kwargs["input"], "->", files_found(call))
    print("answer:", result.response.content)
    print()

--- query-engine tool
search: Ticket 10 customer student discount invoice add now digital album -> ['policy_student_discount.md', 'ticket_10_chile_student_discount.md']
answer: No. The student discount can’t be added once an invoice exists, and it doesn’t apply to digital albums. Ticket 10 was closed with no credit.



--- retriever tool
search: Ticket 10 customer student discount add to invoice digital album -> ['policy_student_discount.md', 'ticket_10_chile_student_discount.md']
answer: No. Ticket 10’s invoice already exists, and the student discount is applied only at checkout—not added afterward. Even with a verified student email, the 10% discount applies only to physical items; it does not cover a digital album.



In our run each agent made one search, and it found `policy_student_discount.md` beside ticket 10. Both answered both parts: no, the discount is applied at checkout only, and no, digital albums are not discounted. The question already names the student discount, so the agent's first search found the policy by its own words; the ticket's pointer (*Policy: student discount is applied at checkout only*) was not needed. Ticket 06's question never names the damaged-media policy: only the ticket does, and that is where a sentence can lose it.

## Going further

Three more things to try, each with a cell to run: count every model call, split a long document, and watch an
agent's events.

### Count every model call, including those inside a tool

`result.tool_calls` shows the searches, not the model calls. LlamaIndex reports every call to its **instrumentation**
system: each model call sends events to a dispatcher, and a handler we register receives them. `LLMChatEndEvent`
comes at the end of each chat call and carries the response, with the token counts the API reported in
`response.raw.usage`.

Two details: an agent streams its replies by default, and a streamed reply comes without usage, so the agents below
are built with `streaming=False`. And ⚠️ run the handler cell once: running it again adds a second handler, and every
call is then counted twice (restart the kernel to undo it).

In [14]:
from llama_index.core.instrumentation import get_dispatcher
from llama_index.core.instrumentation.event_handlers import BaseEventHandler
from llama_index.core.instrumentation.events.llm import LLMChatEndEvent

CALLS = []  # one entry per model call: input and output tokens, as the API reported them


class UsageLog(BaseEventHandler):
    def handle(self, event, **kwargs):
        if isinstance(event, LLMChatEndEvent):
            usage = event.response.raw.usage
            CALLS.append({"input": usage.prompt_tokens, "output": usage.completion_tokens})


get_dispatcher().add_event_handler(UsageLog())

Now one more run of each agent, each counted from the log. The cost uses the prices in `models.env`, in US dollars per
million tokens.

In [15]:
PRICE_IN = float(settings["PRICE_INPUT_PER_MILLION"])
PRICE_OUT = float(settings["PRICE_OUTPUT_PER_MILLION"])

print(f"{'tool':18} {'searches':>8} {'model calls':>11} {'input':>6} {'output':>6} {'cost $':>9}")
for label, tool in [("query-engine tool", qe_tool), ("retriever tool", rt_tool)]:
    start = len(CALLS)
    agent = FunctionAgent(tools=[tool], system_prompt=PROMPT, streaming=False)
    result = await agent.run(user_msg=QUESTION)
    calls = CALLS[start:]
    tokens_in = sum(call["input"] for call in calls)
    tokens_out = sum(call["output"] for call in calls)
    cost = (tokens_in * PRICE_IN + tokens_out * PRICE_OUT) / 1_000_000
    print(f"{label:18} {len(result.tool_calls):>8} {len(calls):>11} {tokens_in:>6} {tokens_out:>6} {cost:>9.6f}")

tool               searches model calls  input output    cost $


query-engine tool         2           5   1386    219  0.000248


retriever tool            2           3   1101     96  0.000158


In our run each agent searched twice. The query-engine-tool agent made 5 model calls and the retriever-tool agent 3: the 2 extra calls are the ones inside the tool, one per search, which `result.tool_calls` does not show. Both runs together cost well under a tenth of a cent at the prices in `models.env`; the gap grows with every search, and with long files the engine's calls read far more text.

### Split a long document into nodes

Our files stay one node each. A long document does not: the index's default splitter, `SentenceSplitter`, cuts text
into pieces of up to 1,024 tokens (a token is the unit a model reads, about three quarters of an English word), and
tries to end each piece at the end of a sentence. To see a split, we join the 20 policies into one handbook in memory
(nothing is written to `data/`) and split it with a small piece size, 128 tokens, chosen only to make the cuts easy to
see. `Document(text=...)` builds the handbook as one document from a string, and `chunk_overlap=20` lets
neighbouring pieces share up to 20 tokens of text at each cut.

In [16]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

policies = [document for document in documents if document.metadata["file_name"].startswith("policy")]
handbook = Document(text="\n\n".join(document.text for document in policies))

print("default piece size:", SentenceSplitter().chunk_size, "tokens")
print("handbook at the default size:", len(SentenceSplitter().get_nodes_from_documents([handbook])), "nodes")

pieces = SentenceSplitter(chunk_size=128, chunk_overlap=20).get_nodes_from_documents([handbook])
print("handbook at 128 tokens:", len(pieces), "nodes")
print()
print(pieces[1].text[:200], "...")
print("links:", [kind.name for kind in pieces[1].relationships])

default piece size: 1024 tokens
handbook at the default size: 2 nodes
handbook at 128 tokens: 16 nodes

# Cancellations

A physical order can be cancelled until it ships. After it ships, use the return policy.

A digital order can be cancelled only before the download starts. After the first byte, it is ...
links: ['SOURCE', 'PREVIOUS', 'NEXT']


The handbook is longer than one 1,024-token piece, so at the default size it becomes 2 nodes, and at 128 tokens 16. The second small piece begins at the heading *# Cancellations*, the start of the second policy: the splitter ended the first piece at the end of a sentence instead of cutting one in half. Besides `SOURCE` (the handbook), a piece cut from a longer document has `PREVIOUS` and `NEXT` links to its neighbours, so code can fetch the text around a search result. Smaller pieces pay off when documents are long and mix topics; the shop's files are short and hold one topic each, so this notebook kept one node per file.

### Watch the agent's events as they happen

`await agent.run(...)` waits for the end. `agent.run(...)` without `await` returns a **handler** at once, and
`handler.stream_events()` yields the run's events as they happen: `ToolCall` when the model asks for a tool,
`ToolCallResult` when the tool has answered, and `AgentOutput` for each model reply. `async for` reads them one by
one; `await handler` then gives the same result as before.

In [17]:
from llama_index.core.agent.workflow import AgentOutput, ToolCall, ToolCallResult

handler = rt_agent.run(user_msg=QUESTION)
async for event in handler.stream_events():
    if isinstance(event, ToolCallResult):
        print("ToolCallResult:", files_found(event))
    elif isinstance(event, ToolCall):
        print("ToolCall:      ", event.tool_kwargs["input"])
    elif isinstance(event, AgentOutput):
        print("AgentOutput:   ", "asks for a tool" if event.tool_calls else "the answer")
result = await handler

AgentOutput:    asks for a tool
ToolCall:       Ticket 06 replacement arrived warped what was offered next, and refund processing timeline. Search ticket 06 and relevant policy.


ToolCallResult: ['ticket_06_germany_warped_vinyl.md', 'ticket_16_overpayment_note.md']


AgentOutput:    asks for a tool
ToolCall:       shop damaged media policy replacement warped vinyl second replacement/refund options and refund processing time duration.


ToolCallResult: ['policy_damaged_media.md', 'ticket_06_germany_warped_vinyl.md']


AgentOutput:    asks for a tool
ToolCall:       refund timing how long does refund take processing days damaged media refund shop policy.


ToolCallResult: ['policy_refunds.md', 'policy_support_hours.md']


AgentOutput:    the answer


In our run: three times the model asked for a tool (`ToolCall`, then its `ToolCallResult` with the files found), then the answer. The second search found `policy_damaged_media.md`, the third the refunds policy. This is the loop that `await agent.run(...)` hides, one event at a time; a chat page would show these events while the agent works.

## Resources

- [LlamaIndex: SimpleDirectoryReader](https://developers.llamaindex.ai/python/framework/module_guides/loading/simpledirectoryreader/):
  reading a folder, and the `file_metadata` argument
- [LlamaIndex: query engines](https://developers.llamaindex.ai/python/framework/module_guides/deploying/query_engine/)
  and [response modes](https://developers.llamaindex.ai/python/framework/module_guides/deploying/query_engine/response_modes/):
  how a query engine turns retrieved nodes into an answer
- [LlamaIndex: tools](https://developers.llamaindex.ai/python/framework/module_guides/deploying/agents/tools/):
  `QueryEngineTool` and `FunctionTool`; the [`RetrieverTool` reference](https://developers.llamaindex.ai/python/framework-api-reference/tools/retriever/)
- [LlamaIndex: agents](https://developers.llamaindex.ai/python/framework/module_guides/deploying/agents/):
  `FunctionAgent` and its run
- [LlamaIndex example: Simple Vector Store](https://developers.llamaindex.ai/python/examples/vector_stores/simpleindexdemo/):
  a query engine limited by `MetadataFilters`
- [LlamaIndex: node parser modules](https://developers.llamaindex.ai/python/framework/module_guides/loading/node_parsers/modules/):
  `SentenceSplitter` and the other splitters
- [LlamaIndex: instrumentation](https://developers.llamaindex.ai/python/framework/module_guides/observability/instrumentation/):
  events and handlers
- [LlamaIndex: streaming agent events](https://developers.llamaindex.ai/python/framework/understanding/agent/streaming/):
  `stream_events`, `ToolCall` and `ToolCallResult`

Back to [notebook 1](../14a_meet_llamaindex.ipynb). Next:
[Lab 15: How to build a customer-support agent over a database](../../15_Support_Desk_Agent/README.md).